In [1]:
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

In [2]:
import pandas as pd

columns = ['id','country','Label','Text']
import pandas as pd
df = pd.read_csv(
    "/kaggle/input/datasets/jp797498e/twitter-entity-sentiment-analysis/twitter_training.csv",
    header=None
)

print(df.shape)


df.head(5)

(74682, 4)


,0,1,2,3
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       74682 non-null  int64 
 1   1       74682 non-null  object
 2   2       74682 non-null  object
 3   3       73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [4]:
path = "/kaggle/input/datasets/jp797498e/twitter-entity-sentiment-analysis/twitter_training.csv"
df = pd.read_csv(path, header=None, names=["id", "entity", "sentiment", "text"])

df = df.dropna(subset=["text"])
print(df["sentiment"].value_counts())

sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64


In [5]:
df = df[df["sentiment"] != "Irrelevant"].reset_index(drop=True)
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentiment"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


In [6]:
df["label"].value_counts()

label
0    22358
2    20655
1    18108
Name: count, dtype: int64

In [7]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True,  max_length=128)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize, batched=True)
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

Map:   0%|          | 0/48896 [00:00<?, ? examples/s]

Map:   0%|          | 0/12225 [00:00<?, ? examples/s]

In [10]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
training_args = TrainingArguments(
    output_dir="./bert_twitter",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [13]:
print(df["sentiment"].unique())
print(df["label"].unique())
print(df["label"].min(), df["label"].max())
print(bert_model.config.num_labels)
print(train_dataset["labels"][:10])
print(max(train_dataset["labels"]), min(train_dataset["labels"]))

['Positive' 'Neutral' 'Negative']
[2 1 0]
0 2
3
[1, 2, 1, 2, 0, 1, 1, 0, 0, 0]
2 0


In [14]:
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.885618,0.846159,0.835828
2,0.470331,0.579944,0.895624
3,0.229360,0.527756,0.916155


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=4584, training_loss=0.6932744895600523, metrics={'train_runtime': 2197.0458, 'train_samples_per_second': 66.766, 'train_steps_per_second': 2.086, 'total_flos': 9648895254921216.0, 'train_loss': 0.6932744895600523, 'epoch': 3.0})

In [15]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.5277557969093323,
 'eval_accuracy': 0.9161554192229039,
 'eval_runtime': 62.8545,
 'eval_samples_per_second': 194.497,
 'eval_steps_per_second': 6.093,
 'epoch': 3.0}

In [16]:
preds = trainer.predict(test_dataset)
y_pred = preds.predictions.argmax(axis=1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [20]:

from sklearn.metrics import accuracy_score, classification_report
bert_accuracy = accuracy_score(test_dataset["labels"], y_pred)
print(bert_accuracy)

0.9161554192229039


In [21]:
print(classification_report(
    test_dataset["labels"], y_pred,
    target_names=le.classes_
))

              precision    recall  f1-score   support

    Negative       0.93      0.93      0.93      4472
     Neutral       0.91      0.90      0.90      3622
    Positive       0.91      0.92      0.91      4131

    accuracy                           0.92     12225
   macro avg       0.92      0.92      0.92     12225
weighted avg       0.92      0.92      0.92     12225



In [22]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(test_dataset["labels"], y_pred))

[[4162  147  163]
 [ 165 3255  202]
 [ 160  188 3783]]


In [24]:
import time

start = time.time()
preds = trainer.predict(test_dataset)
bert_infer_time = time.time() - start
y_pred = preds.predictions.argmax(axis=1)
bert_accuracy = accuracy_score(test_dataset["labels"], y_pred)
print(bert_accuracy, bert_infer_time)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


0.9161554192229039 62.18518924713135


In [25]:
distil_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_distil(batch):
    return distil_tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [26]:
train_distil = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_distil, batched=True)
test_distil = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize_distil, batched=True)

train_distil = train_distil.rename_column("label", "labels")
test_distil = test_distil.rename_column("label", "labels")

Map:   0%|          | 0/48896 [00:00<?, ? examples/s]

Map:   0%|          | 0/12225 [00:00<?, ? examples/s]

In [27]:
distil_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
distil_args = TrainingArguments(
    output_dir="./distil_twitter",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

In [31]:
distil_trainer = Trainer(
    model=distil_model,
    args=distil_args,
    train_dataset=train_distil,
    eval_dataset=test_distil,
    compute_metrics=compute_metrics
)

start = time.time()
distil_trainer.train()
distil_train_time = time.time() - start

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.947443,0.940970,0.814151
2,0.632107,0.628467,0.884581
3,0.368610,0.592821,0.899141


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [32]:
start = time.time()
preds = distil_trainer.predict(test_distil)
distil_infer_time = time.time() - start
y_pred_distil = preds.predictions.argmax(axis=1)
distil_accuracy = accuracy_score(test_distil["labels"], y_pred_distil)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [33]:
results = pd.DataFrame({
    "Model": ["BERT", "DistilBERT"],
    "Accuracy": [bert_accuracy, distil_accuracy],
    "Inference_Time_s": [bert_infer_time, distil_infer_time]
})
print(results)

        Model  Accuracy  Inference_Time_s
0        BERT  0.916155         62.185189
1  DistilBERT  0.899141         30.815061


In [34]:
import numpy as np
from scipy.special import softmax

preds = trainer.predict(test_dataset)
probs = softmax(preds.predictions, axis=1)
y_true = np.array(test_dataset["labels"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [35]:
print(df["sentiment"].value_counts())
print(dict(zip(le.classes_, le.transform(le.classes_))))

sentiment
Negative    22358
Positive    20655
Neutral     18108
Name: count, dtype: int64
{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


In [36]:
from sklearn.metrics import f1_score

minority = 1  # change if the counts say otherwise

for t in [0.3, 0.5, 0.7]:
    others = probs.copy()
    others[:, minority] = -1
    fallback = others.argmax(axis=1)
    y_pred = np.where(probs[:, minority] >= t, minority, fallback)
    f1 = f1_score(y_true, y_pred, labels=[minority], average=None)[0]
    print(t, "minority F1:", round(f1, 4))

0.3 minority F1: 0.8998
0.5 minority F1: 0.9048
0.7 minority F1: 0.8966


In [37]:
from sklearn.metrics import classification_report

best_t = 0.5  # set to the winner
others = probs.copy()
others[:, minority] = -1
y_pred = np.where(probs[:, minority] >= best_t, minority, others.argmax(axis=1))
print(classification_report(y_true, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

    Negative       0.91      0.94      0.92      4472
     Neutral       0.93      0.88      0.90      3622
    Positive       0.90      0.92      0.91      4131

    accuracy                           0.91     12225
   macro avg       0.92      0.91      0.91     12225
weighted avg       0.91      0.91      0.91     12225



In [38]:
import json

trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

with open("./best_model/labels.json", "w") as f:
    json.dump(list(le.classes_), f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [39]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained("./best_model")
mdl = AutoModelForSequenceClassification.from_pretrained("./best_model").eval()
labels = json.load(open("./best_model/labels.json"))

def predict(text):
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = mdl(**inputs).logits[0].numpy()
    probs = softmax(logits)
    idx = int(probs.argmax())
    return {"label": labels[idx], "confidence": round(float(probs[idx]), 3)}

print(predict("I love this game!"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'label': 'Positive', 'confidence': 0.997}
